In [1]:
import os
import gradio as gr
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import urllib.request
from urllib.parse import quote

import config

C:\Users\HKIT\AppData\Local\Temp\ipykernel_12644\2275509073.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [3]:
# PDF 다운로드

urllib.request.urlretrieve(quote("https://raw.githubusercontent.com/ai-agent-kr/agent-tutorial/main/Ch02/2020_경제금융용어 700선_게시.pdf", safe=":/"), filename="2020_경제금융용어 700선_게시.pdf")

('2020_경제금융용어 700선_게시.pdf', <http.client.HTTPMessage at 0x15adc8c8410>)

In [5]:
# PDF 로드 + 청크 확인 (texts[15])

loader = PyPDFLoader("2020_경제금융용어 700선_게시.pdf")
texts = loader.load_and_split()
print('문서의 수:', len(texts))

문서의 수: 366


In [6]:
# 임의의 15번 청크를 출력해 형식을 확인

texts[15]

Document(metadata={'producer': 'Smart PDF Imposition 1.0', 'creator': 'Smart PDF Imposition 1.0', 'creationdate': 'January 31, 2018 16:21', 'author': 'INSUNG DATA INC.', 'creationdate--text': 'January 31, 2018 16:21', 'moddate': 'January 31, 2018 16:21', 'moddate--text': 'January 31, 2018 16:21', 'subject': 'CoreImposition PDF 1.0 Output', 'title': '°æÁ¦±ÝÀ¶¿ë¾î 700¼±-ÃÖÁ¾.PDF', 'source': '2020_경제금융용어 700선_게시.pdf', 'total_pages': 371, 'page': 19, 'page_label': '20'}, page_content='3\nㄱ \n신용을 모두 포괄하는 것으로 크게 가계대출과 판매신용으로 구분된다. 현재 가계신용통\n계는 2002년 말 잔액부터 분기별로 제공되고 있다. 가계신용통계는 우리나라 가계부문의 \n부채 규모 및 변동 등을 파악하는 데 널리 활용되고 있다 .\n가계처분가능소득 \n가계처분가능소득(PDI; Personal Disposable Income)은 가계가 맘대로 소비와 저축\n으로 처분할 수 있는 소득을 의미한다. 흔히 국민들의 생활수준을 파악해 볼 수 있는 \n지표로 1인당 GNI가 널리 쓰이고 있으나 국민총소득에는 가계 뿐 아니라 기업 금융기관 \n정부가 벌어 들인 소득이 모두 포함되어 있다. 따라서 기업과 금융기관 등이 가계부문 \n보다 더 많은 소득을 벌어 1인당 국민총소득(GNI)이 높아진 경우에는 가계가 느끼는 \n체감경기는 전체 경기와 괴리가 있게 된다. 1인당 가계총처분가능소득(PGDI; Personal \nGross Disposable Income)은 가계부문의 총처분가능소득을 연앙인구로 나누

In [7]:
# 본문 추출 (.page_content) & 0번 청크

print(texts[15].page_content)

3
ㄱ 
신용을 모두 포괄하는 것으로 크게 가계대출과 판매신용으로 구분된다. 현재 가계신용통
계는 2002년 말 잔액부터 분기별로 제공되고 있다. 가계신용통계는 우리나라 가계부문의 
부채 규모 및 변동 등을 파악하는 데 널리 활용되고 있다 .
가계처분가능소득 
가계처분가능소득(PDI; Personal Disposable Income)은 가계가 맘대로 소비와 저축
으로 처분할 수 있는 소득을 의미한다. 흔히 국민들의 생활수준을 파악해 볼 수 있는 
지표로 1인당 GNI가 널리 쓰이고 있으나 국민총소득에는 가계 뿐 아니라 기업 금융기관 
정부가 벌어 들인 소득이 모두 포함되어 있다. 따라서 기업과 금융기관 등이 가계부문 
보다 더 많은 소득을 벌어 1인당 국민총소득(GNI)이 높아진 경우에는 가계가 느끼는 
체감경기는 전체 경기와 괴리가 있게 된다. 1인당 가계총처분가능소득(PGDI; Personal 
Gross Disposable Income)은 가계부문의 총처분가능소득을 연앙인구로 나누어 계산한 
지표로 가계의 구매력을 가장 정확히 가늠해 볼 수 있는 소득지표이다 . 
 연관검색어 : 국민총소득 (GNI)
가교은행
청산 대상 금융기관의 자산, 부채를 임시로 넘겨받아 예금, 출금 등의 업무를 대행하고 
합병, 채권채무관계 조정 등 후속조치를 수행하는 은행을 말한다. 금융기관이 파산한 경우 
예금보험제도는 청산, 매각, 자산부채승계, 가교은행을 통한 인수 등의 수단을 통해 부실 
금융기관을 처리한다. 이 중 가교은행을 통한 인수방식은 파산은행의 처리에 상당한 시간이 
소요될 것으로 예상할 경우 사용되며 새로운 은행을 설립하여 자산, 부채를 포괄승계하도록 
하고, 인수 희망자를 물색하는 등의 조치를 취하게 된다. 가교은행을 이용한 처리방식은 
금융기관 파산에 따른 충격을 완화하고 시간적인 여유를 확보할 수 있는 것이 장점이다. 
임무를 마치면 정리대상 금융기관과 마찬가지로 없어지는 한시적 기관이다.
가동률
생산능력 대비 생산실적의 백분율(생산실적/생산능력×100)로

In [8]:
# 0번 청크 실행 결과

print(texts[0].page_content)

iii
찾아보기
한국은행은 국민들이 경제 및 금융에 대한 이해도를 높이고 경제에 관한 합리적인 
의사결정 능력을 키울 수 있도록 현장 경제교육, 온라인 경제교육, 경제교육 콘텐츠 
개발 등 대국민 경제교육을 다양하게 수행해 오고 있습니다 . 
이의 일환으로 2018년 
 경제금융용어 700선
 을 발간하였는데 그간 동 책자에 대한 
수요가 꾸준히 늘어남에 따라 이번에 추가로 발간하게 되었습니다 .
지난번 내용과 같이 통화정책, 실물경제, 금융안정, 지급결제 등 한국은행 주요 
업무를 이해하는데 필요한 전문 용어와 경제·금융 흐름을 이해하는데 도움이 되는 
시사 경제금융 용어들을 수록하였습니다. 용어해설은 개념과 도입 배경, 의미, 적용 
사례 등을 담아 쉽게 이해할 수 있도록 하였습니다. 또한 e-book으로도 제작하여 
독자들의 편의성과 가독성을 높였습니다 . 
코로나 19로 이전에 경험하지 못한 경제위기를 겪으면서 경제·금융 현상에 대한 
일반인들의 관심도가 이전에 비해 높아졌습니다. 아무쪼록 이 책자가 한국은행의 주요 
정책과 국내외 경제를 이해하는 데 조금이나마 도움이 되기를 바라겠습니다. 끝으로 
경제금융용어 700선
  집필에 참여해주신 경제교육실 교수님들과 용어 집필에 좋은 
의견을 주신 관련부서 여러분께 감사드립니다 .
2020년 8월
한국은행 경제교육실장 박 철 원
머 리 말


In [9]:
# 청크 점검 - 머리말 목차 확인

print(texts[5].page_content)

vi
경제금융용어 700선
ㅂ
바이오인증 ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・118
바젤은행감독위원회/
바젤위원회(BCBS) ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・118
발행시장 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・119
발행중지화폐/유통정지화폐 ・・・・・・・・・・・・・・・120
방카슈랑스 ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・120
배당할인모형 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・121
밴드웨건효과 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・121
뱅 크 런 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・122
범위의 경제 ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・122
법률리스크 ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・122
베블런효과 ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・123
변동금리 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・123
변동금리부채권(FRN) ・ ・ ・・・・・・・・・・・・・・・・・・・・・・・123
보기화폐(견양화폐) ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・124
보완자본(Tier 2) ・・・・・・・・・・・

In [10]:
print(texts[12].page_content)

xiii
찾아보기❙
환매조건부매매/RP/Repo ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・331
환 어 음 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・332
환율조작국 ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・332
환전영업자(환전상) ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・333
환차손/환차익 ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・333
황금낙하산 ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・ ・334
회 사 채 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・334
회수의문 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・335
후방연쇄효과 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・335
후순위금융채 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・335
후행종합지수 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・336
ABC
B e y o n d  G D P ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・337
BIS 자기자본비율 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・338
CAMEL-R시스템/ROCA시스템/
C A C R E L 시 스 템 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・338
C L S 은 행 ・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・・339
C M O ・・・・・・・・・・・・・・・・

In [14]:
# 청크 점검 - 용어 설명 시작 (texts[13])

print(texts[13].page_content)

1
ㄱ 
ㄱ
가계부실위험지수(HDRI)
가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 
부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는 원리금상
환비율(DSR; Debt Service Ratio)과 자산 측면에서 평가하는 부채/자산비율(DTA; Debt 
To Asset Ratio)을 결합하여 산출한 지수이다. 가계부실위험지수는 가구의 DSR과 DTA가 
각각 40%, 100%일 때 100의 값을 갖도록 설정되어 있으며, 동 지수가 100을 초과하는 
가구를 ‘위험가구’로 분류한다. 위험가구는 소득 및 자산 측면에서 모두 취약한 ‘고위험가구’, 
자산 측면에서 취약한 ‘고DTA가구’, 소득 측면에서 취약한 ‘고DSR가구’로 구분할 수 있다. 
다만 위험 및 고위험 가구는 가구의 채무상환능력 취약성 정도를 평가하기 위한 것이며 
이들 가구가 당장 채무상환 불이행, 즉 임계상황에 직면한 것을 의미하지 않는다 .
 연관검색어 : 총부채원리금상환비율(DSR)
가계수지
가정에서 일정 기간의 수입(명목소득)과 지출을 비교해서 남았는지 모자랐는지를 
표시한 것을 가계수지(household's total income and expenditure)라 한다. 가계수지가 
가계수지 ∙


In [15]:
# 앞부분(머리말, 목차) 제거

texts = texts[13:]
print('줄어든 문서의 개수:', len(texts))

줄어든 문서의 개수: 353


In [16]:
# 새 0번 청크 확인

print('첫 번째 문서 출력:', texts[0])

첫 번째 문서 출력: page_content='1
ㄱ 
ㄱ
가계부실위험지수(HDRI)
가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 
부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는 원리금상
환비율(DSR; Debt Service Ratio)과 자산 측면에서 평가하는 부채/자산비율(DTA; Debt 
To Asset Ratio)을 결합하여 산출한 지수이다. 가계부실위험지수는 가구의 DSR과 DTA가 
각각 40%, 100%일 때 100의 값을 갖도록 설정되어 있으며, 동 지수가 100을 초과하는 
가구를 ‘위험가구’로 분류한다. 위험가구는 소득 및 자산 측면에서 모두 취약한 ‘고위험가구’, 
자산 측면에서 취약한 ‘고DTA가구’, 소득 측면에서 취약한 ‘고DSR가구’로 구분할 수 있다. 
다만 위험 및 고위험 가구는 가구의 채무상환능력 취약성 정도를 평가하기 위한 것이며 
이들 가구가 당장 채무상환 불이행, 즉 임계상황에 직면한 것을 의미하지 않는다 .
 연관검색어 : 총부채원리금상환비율(DSR)
가계수지
가정에서 일정 기간의 수입(명목소득)과 지출을 비교해서 남았는지 모자랐는지를 
표시한 것을 가계수지(household's total income and expenditure)라 한다. 가계수지가 
가계수지 ∙' metadata={'producer': 'Smart PDF Imposition 1.0', 'creator': 'Smart PDF Imposition 1.0', 'creationdate': 'January 31, 2018 16:21', 'author': 'INSUNG DATA INC.', 'creationdate--text': 'January 31, 2018 16:21', 'moddate': 'January 31, 2018 16:21', 'moddate--text': 'January 31, 2018 16:21', 'subject': 'CoreImposition PDF 1.0 Output', 'ti

In [17]:
# 맺음말(마지막 청크) 제거

print('마지막 문서 출력:', texts[-1])

마지막 문서 출력: page_content='경제금융용어 700선
발행인 이주열
편집인 박철원
발행처 한국은행(www.bok.or.kr)
서울특별시 중구 세종대로 67(태평로 2가)
발행일 2020년 8월 3일
인  쇄 ㈜제일프린테크 
 본 자료는 한국은행 홈페이지(http://www.bok.or.kr>경제교육>온라인학습>일반인) 에서 
다운로드 받으실 수 있습니다.
 이 책자에 대한 문의는 한국은행 경제교육실 경제교육기획팀(02-759-5618)으로 연락 
하여 주시기 바랍니다.
 
 경제금융용어 700선
 은 
① 정부간행물판매센터(☎ 02-734-6818, http://www.gpcbooks.co.kr) 
② 한국경제서적(☎ 02-737-7498)
③ 경제서적(☎ 02-736-0640, http://kj-book.co.kr) 또는 주요 서점 등에서 매권당 
8,000원에 판매하고 있습니다.
ISBN 979-11-5538-393-3 03320' metadata={'producer': 'Smart PDF Imposition 1.0', 'creator': 'Smart PDF Imposition 1.0', 'creationdate': 'January 31, 2018 16:21', 'author': 'INSUNG DATA INC.', 'creationdate--text': 'January 31, 2018 16:21', 'moddate': 'January 31, 2018 16:21', 'moddate--text': 'January 31, 2018 16:21', 'subject': 'CoreImposition PDF 1.0 Output', 'title': '°æÁ¦±ÝÀ¶¿ë¾î 700¼±-ÃÖÁ¾.PDF', 'source': '2020_경제금융용어 700선_게시.pdf', 'total_pages': 371, 'page': 369, 'page_label': '370'}


In [20]:
# 마지막 청크 제거 후 개수 확인

texts = texts[:-1]
print('마지막 데이터 제거 후 문서의 개수:', len(texts))

마지막 데이터 제거 후 문서의 개수: 352


In [21]:
# 벡터 데이터베이스 적재 (Chroma)

embedding = OpenAIEmbeddings()

vectordb = Chroma.from_documents(documents=texts, embedding=embedding)

print(vectordb._collection.count())

352


In [ ]:
# 호출 가능한 키 확인

for key in vectordb._collection.get():
    print(key)

ids
embeddings
documents
uris
included
data
metadatas


In [27]:
# documents 불러오기

documents = vectordb._collection.get()['documents']
print('문서의 개수:', len(documents))
print('첫 번째 문서 출력:', documents[0])

문서의 개수: 352
첫 번째 문서 출력: 1
ㄱ 
ㄱ
가계부실위험지수(HDRI)
가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 
부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는 원리금상
환비율(DSR; Debt Service Ratio)과 자산 측면에서 평가하는 부채/자산비율(DTA; Debt 
To Asset Ratio)을 결합하여 산출한 지수이다. 가계부실위험지수는 가구의 DSR과 DTA가 
각각 40%, 100%일 때 100의 값을 갖도록 설정되어 있으며, 동 지수가 100을 초과하는 
가구를 ‘위험가구’로 분류한다. 위험가구는 소득 및 자산 측면에서 모두 취약한 ‘고위험가구’, 
자산 측면에서 취약한 ‘고DTA가구’, 소득 측면에서 취약한 ‘고DSR가구’로 구분할 수 있다. 
다만 위험 및 고위험 가구는 가구의 채무상환능력 취약성 정도를 평가하기 위한 것이며 
이들 가구가 당장 채무상환 불이행, 즉 임계상황에 직면한 것을 의미하지 않는다 .
 연관검색어 : 총부채원리금상환비율(DSR)
가계수지
가정에서 일정 기간의 수입(명목소득)과 지출을 비교해서 남았는지 모자랐는지를 
표시한 것을 가계수지(household's total income and expenditure)라 한다. 가계수지가 
가계수지 ∙


In [ ]:
# embedding Vector 불러오기

embeddings = vectordb._collection.get(include=['embeddings'])['embeddings']
print('임베딩 벡터의 개수:', len(embeddings))

임베딩 벡터의 개수: 352


In [30]:
# 첫 번째 임베딩 값을 확인

print('첫 번째 문서의 임베딩 값 출력:', embeddings[0])
print('첫 번째 문서의 임베딩 값의 길이:', len(embeddings[0]))

첫 번째 문서의 임베딩 값 출력: [-0.0077088  -0.0075166   0.014182   ... -0.01861645 -0.00669286
 -0.00903364]
첫 번째 문서의 임베딩 값의 길이: 1536


In [41]:
# db.get(include=['embeddings']) -> 딕셔너리 반환 확인 코드

dic = vectordb.get(include=['embeddings'])
print(dic.keys())
print(dic)

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])
{'ids': ['305a958c-9781-4a64-867e-4c320e833d25', 'aa9d136f-5e34-428e-aa2e-f8a612dc0814', '15d8b2e4-664b-4b68-af78-a835b8c0a008', '26e991ba-b9c7-4f3d-991a-02c29445a42c', 'e3991673-a55b-473b-8aa4-e5f57e388ab3', 'f9e25c56-ea52-40a8-919d-032d47217857', 'a2285f18-017d-499e-b83d-f07283f0b674', 'd7b405ec-6fba-446e-840f-507189aae382', 'daab6963-4b56-4fa1-b0dc-fa4df42604b9', 'e84eb8d5-7907-48c1-8e08-33399e14bcab', '5a17722e-7a05-4cbc-b5ca-eac7fccd00b7', 'a3b177cc-2ea8-45f6-bccc-91b796ea2761', '8696d843-f2fe-4537-961b-18d28be0ebec', '36933935-4f1a-4f7c-822e-6c90b705175f', '3671fe29-ed60-40c8-8c68-d94669bb665c', '18864da2-15cf-4e6f-a636-c8b2e7a10563', '6b1cd958-a95e-4fbd-963b-eda54c72c99a', 'ea96f1b3-e8a3-4fae-b1ea-47237a53545a', '72bbcc9e-7d84-4a12-b58b-bb2d52277e59', '54e2ec31-ed93-47ee-bcd8-35db19784b94', '88a7ebb6-991e-4062-a759-f24bb164f80e', '9e2469c2-431f-412c-8a98-a17831666d81', '8e9cc481-60e9-4511-bc08

In [45]:
# [...]['embeddings'] -> 딕셔너리 에서 'embeddings' 값만 꺼냄

e = vectordb.get(include=['embeddings'])['embeddings']
e.shape
e[0]

array([-0.0077088 , -0.0075166 ,  0.014182  , ..., -0.01861645,
       -0.00669286, -0.00903364], shape=(1536,))

In [ ]:
# 출처 (metadatas)확인

metadatas = vectordb._collection.get()['metadatas']
print('metadatas의 개수:', len(metadatas))
print('첫 번째 문서의 출처:', metadatas[0])

metadatas의 개수: 352
첫 번째 문서의 출처: {'subject': 'CoreImposition PDF 1.0 Output', 'total_pages': 371, 'producer': 'Smart PDF Imposition 1.0', 'source': '2020_경제금융용어 700선_게시.pdf', 'creationdate': 'January 31, 2018 16:21', 'page': 17, 'moddate--text': 'January 31, 2018 16:21', 'page_label': '18', 'creationdate--text': 'January 31, 2018 16:21', 'creator': 'Smart PDF Imposition 1.0', 'moddate': 'January 31, 2018 16:21', 'author': 'INSUNG DATA INC.', 'title': '°æÁ¦±ÝÀ¶¿ë¾î 700¼±-ÃÖÁ¾.PDF'}


In [67]:
# 검색기 선언 (상위 2개)

retriever = vectordb.as_retriever(search_kwargs={"k":2})

In [68]:
# 검색 실행
docs = retriever.invoke("비트코인이 궁금해")
print('유사 문서 개수:', len(docs))
print('--' * 20)
print('첫 번째 유사 문서:', docs[0])
print('두 번째 유사 문서:', docs[1])

유사 문서 개수: 2
----------------------------------------
첫 번째 유사 문서: page_content='139
ㅂ 
비트코인
비트코인(bitcoin)은 가상통화(암호통화)이자 디지털 지급시스템이다. 비트코인 시스템
은 중앙 저장소 또는 단일 관리자가 없기 때문에 최초의 탈중앙화된 디지털통화라고 불린다. 
이는 사토시 나카모토라는 사람(집단)에 의해 만들어져서 2009년 개방형 소프트웨어로 
배포되었다. 이 시스템은 공유형(peer-to-peer)이며, 거래는 중개자 없이 블록체인 소프트
웨어를 이용하는 참여자(nodes) 사이에 직접 이뤄진다. 이런 거래들은 공유(P2P) 네트워크
상 참여자의 작업증명(proof-of-work)을 통해 검증되고 공개된 분산원장인 블록체인에 
기록된다. 승인된 거래의 새 그룹인 한 블록은 대략 10분에 한 번씩 만들어져서 블록체인에 
추가되고 신속하게 모든 참여자에게 보내어진다. 비트코인은 대규모 전기를 사용하는 
컴퓨터 처리 능력(power)을 활용해서 이뤄지는 기록보관 서비스인 채굴(mining)에 대한 
보상으로 발행되고 있으며 다른 통화･상품･용역 등과 교환되어가고 있다. 중앙은행이 
발행한 법정화폐가 아닌 비트코인은 비트코인 플랫폼에서 거래되며 투자대상으로서도 
관심을 받고 있다. 하지만 급등한 가격 및 심한 변동 폭으로 거품논란이 크다. 또한 익명성으
로 자금세탁 등 불법거래에 악용될 소지가 많아 중국 등 일부 국가에서 비트코인 등 가상통
화의 거래를 규제하고 있다. 일본의 경우 비트코인의 거래이익에 대해 세금을 부과한다. 
비트코인은 추가되는 한 블록당 새로 12.5비트코인을 보상하는데(2016.7월 현재), 21만개
가 채굴될 때(대략 4년)마다 그 보상이 반으로 줄어든다. 비트코인의 총량은 21백만개이며 
2140년경 모두 채굴될 것으로 전망된다. 비트코인은 그 시스템의 설계시 그 수량을 한정시
켜 놓았기 때문에 원칙적으로 인플레이션에 의해 가치가 떨어질 수 있는 화폐와 다른

In [ ]:
# 프롬프트 템플릿 만들기

template = """당신은 우리집에서 만든 금융용어를 설명해주는 금융로봇입니다. 노동민 개발자가 만들었습니다. 주어진 검색 결과를 바탕으로 답변하세요.
검색결과에 없는 내용이라면 답변할 수 없다고 하지말고 모르겠으니 딴데가서 알아보라고 하세요. 반말로 친근하고 귀엽게 답변하세요.
{context}

Question:{question}

Answer:
"""
prompt = PromptTemplate.from_template(template)

In [54]:
# ChatGPT 연결 : query_rag()함수

llm = ChatOpenAI(model="gpt-4.1", temperature=0)

def query_rag(question):
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    formatted_prompt = prompt.format(question=question, context=context)
    response = llm.invoke(formatted_prompt)
    return {
        "query":question,
        "result":response.content,
        "source_documents":docs}

In [ ]:
# query_rag 실행

input_text = "디커플링이란 무엇인가?"
chatbot_response = query_rag(input_text)
print(chatbot_response)

{'query': '맛있는 커피?', 'result': '음~ 커피 얘기는 금융용어가 아니라서 내가 잘 모르겠네! 혹시 커피에 대해 궁금한 거 있으면 딴 데 가서 알아보는 게 좋을 것 같아. 그래도 커피 마시면서 금융 공부하면 더 재밌을 수도 있지! ☕️', 'source_documents': [Document(metadata={'creator': 'Smart PDF Imposition 1.0', 'source': '2020_경제금융용어 700선_게시.pdf', 'total_pages': 371, 'page': 201, 'title': '°æÁ¦±ÝÀ¶¿ë¾î 700¼±-ÃÖÁ¾.PDF', 'creationdate--text': 'January 31, 2018 16:21', 'moddate': 'January 31, 2018 16:21', 'page_label': '202', 'subject': 'CoreImposition PDF 1.0 Output', 'creationdate': 'January 31, 2018 16:21', 'producer': 'Smart PDF Imposition 1.0', 'moddate--text': 'January 31, 2018 16:21', 'author': 'INSUNG DATA INC.'}, page_content='185\nㅇ \n가 무척 다양한 옥수수 값 상승의 영향은 과자나 빵 등 식품의 가격에만 국한되지 \n않으며, 설탕의 대체재로 사용되고 옥수수로 만드는 감미료의 가격이 상승하여 이를 \n사용하는 각종 청량음료나 사탕 등의 제품가격에도 영향을 미치게 되었다. 또한, 옥수수\n는 가축사료로 쓰임에 따라 가축을 기르는 비용의 상승으로 이어져 계란, 우유, 베이컨 \n등의 제품가격도 오를 수밖에 없다. 이외에 종이나 옷, 기저귀, 샴푸, 페인트, 크레용 \n등에도 옥수수가 들어가기 때문에 이들 제품의 가격도 오를 가능성이 있다. 이러한 \n애그플레이션의 원인으로는 ① 지구 온난화와 이상 기후 등으로 인한 농작물 생산량 \

In [58]:
# 답변만 추출 : get_chatbot_response()

def get_chatbot_response(input_text):
    chatbot_response = query_rag(input_text)
    return chatbot_response['result'].strip()

In [61]:
# 질문 1

input_text = "너는 뭘 하는 챗봇이니?"
result = get_chatbot_response(input_text)
print(result)

나는 우리집에서 만든 금융용어 설명해주는 귀여운 금융로봇이야! 노동민 개발자가 만들었고, 너가 궁금한 금융용어나 경제용어를 검색 결과를 바탕으로 쉽고 친근하게 설명해주는 역할을 해. 모르는 건 솔직하게 모른다고 하고, 딴 데 가서 알아보라고도 해줄 수 있어. 궁금한 거 있으면 언제든 물어봐! 😊


In [63]:
# 질문 2

input_text = "비트코인에 대해서 궁금할듯 말듯~"
result = get_chatbot_response(input_text)
print(result)

비트코인? 그거 요즘 뉴스에서 많이 들어봤지? 내가 귀엽게 설명해줄게!

비트코인은 인터넷에서만 존재하는 가상통화(암호통화)야. 디지털 지급시스템이기도 하고! 이게 뭐가 특별하냐면, 중앙은행이나 정부 같은 누군가가 관리하지 않고, 전 세계 사람들이 컴퓨터로 서로 거래를 직접 해. 그래서 ‘탈중앙화된 디지털통화’라고 불려.

비트코인은 2009년에 사토시 나카모토라는 정체불명의 사람이 만들었대. 거래할 때는 블록체인이라는 기술을 써서, 거래내역이 모두에게 공개되고, 위조나 해킹이 엄청 어렵게 만들어놨어. 그래서 보안이 빵빵하지!

비트코인은 채굴(mining)이라는 과정을 통해 새로 만들어지는데, 이게 컴퓨터로 엄청 복잡한 계산을 해서 보상으로 비트코인을 받는 거야. 근데 이 보상도 4년마다 반으로 줄어들고, 총 2,100만 개까지만 만들어질 수 있어. 그래서 희소성이 있지!

비트코인은 가격이 엄청 오르락내리락해서 투자 대상으로도 인기 많지만, 변동성이 커서 위험하기도 해. 그리고 익명성이 강해서 불법거래에 쓰일 수도 있어서, 몇몇 나라에서는 규제도 하고 있어.

정리하자면, 비트코인은 인터넷 세상에서만 쓰는 돈인데, 아무나 만들 수 없고, 거래도 투명하게 기록돼서 신기한 점이 많아! 더 궁금한 거 있음 또 물어봐~ 귀엽게 알려줄게!


In [66]:
# Gradio 로 챗봇 UI 구현

# 인터페이스 생성
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="경제금융용어 챗봇")
    msg = gr.Textbox(label="질문을 입력해 주세요.")
    clear = gr.Button("대화 초기화")

    # 챗봇의 답변을 처리하는 함수
    def respond(message, chat_history):
        result = query_rag(message)
        bot_message = result['result']
        chat_history.append({"role":"user","content":message})
        chat_history.append({"role":"assistant","content":bot_message})
        return "", chat_history

    # 입력 제출 시 respond 호출 / 초기화 버튼
    msg.submit(respond, [msg, chatbot], [msg, chatbot]) # 함수, inputs, outputs
    clear.click(lambda: None, None, chatbot, queue=False)
# 인터페이스 실행
demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
